In [ ]:
# ── Setup: put src/ on the path, then build the analysis dataframe through the
#    configurable merger. All data paths live in src/config.py.
import os, sys
_p = os.getcwd()
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, os.path.join(_p, "src")); break
    _p = os.path.dirname(_p)

from data_load import load_data_for_modeling, load_and_clean_accident_data
from topic_vd_analysis import run_topic_vd_analysis
import ruptures as rpt
import os
from typing import Dict, Optional, Tuple, Any
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns
from pathlib import Path

from analysis_data import build_analysis_dataframe

df_acc = build_analysis_dataframe(config="main_0.3")

## Rear end analysis

In [30]:
from rear_end_analysis import run_rear_end_analysis

profile_df, tests_df, geo_df, cp_df = run_rear_end_analysis(
    data=df_acc,
    output_dir="results/rear_end_analysis",
    criterion="bic",
)

[1] Profile comparison saved
[2] Statistical tests saved
[3] Geographic distribution saved
[4] Changepoints saved (7 detected)
 topic  breakpoint_year criterion  mean_rate_before  mean_rate_after direction
     9             2000       bic            41.874          103.639         ↑
     9             2020       bic            83.090           69.095         ↓
    18             2000       bic             4.381           47.191         ↑
    23             1990       bic             7.523           19.312         ↑
    23             2015       bic            15.122           25.379         ↑
    27             1995       bic            20.110           13.132         ↓
    27             2015       bic            13.876           17.445         ↑
[plot] Temporal plot saved
[plot] Geographic plot saved
[5] Label distribution saved — Påkørsel bagfra goes to 34 topics

All outputs written to: results/rear_end_analysis


In [40]:
# show distribution of siuationcode for topic 9, 18, 23, 27
df_acc[df_acc["assigned_topic"].isin([9, 18, 23, 27])].groupby("UHELDSSITUATION").size()

UHELDSSITUATION
11.0         5
12.0         2
23.0         1
32.0         1
40.0         1
50.0         1
98.0         5
111.0      742
112.0      126
140.0    10019
151.0      352
152.0      153
160.0      174
170.0       79
198.0      427
211.0        4
241.0        8
242.0        7
250.0        1
270.0       10
298.0        8
311.0       16
312.0        3
321.0       30
322.0        5
323.0        1
398.0        2
410.0       10
420.0        1
440.0        2
498.0        6
510.0        8
520.0        6
610.0        2
650.0        4
660.0       10
670.0        3
710.0       14
720.0        1
752.0        4
798.0        1
811.0        6
812.0        1
821.0        2
835.0        4
841.0        2
851.0        1
880.0        1
898.0        1
910.0        6
920.0        8
999.0       10
dtype: int64

In [42]:
# how many percent of siuation code 140 are in topic 9, 18, 23, 27  
percent_in_topic_9 = df_acc[(df_acc["UHELDSSITUATION"] == 140) & (df_acc["assigned_topic"] == 9)].shape[0] / df_acc[df_acc["assigned_topic"] == 9].shape[0] * 100
print(f"Percentage of situation code 140 in topic 9: {percent_in_topic_9:.2f}%")


percent_in_topic_18 = df_acc[(df_acc["UHELDSSITUATION"] == 140) & (df_acc["assigned_topic"] == 18)].shape[0] / df_acc[df_acc["assigned_topic"] == 18].shape[0] * 100
print(f"Percentage of situation code 140 in topic 18: {percent_in_topic_18:.2f}%")

percent_in_topic_23 = df_acc[(df_acc["UHELDSSITUATION"] == 140) & (df_acc["assigned_topic"] == 23)].shape[0] / df_acc[df_acc["assigned_topic"] == 23].shape[0] * 100
print(f"Percentage of situation code 140 in topic 23: {percent_in_topic_23:.2f}%")

percent_in_topic_27 = df_acc[(df_acc["UHELDSSITUATION"] == 140) & (df_acc["assigned_topic"] == 27)].shape[0] / df_acc[df_acc["assigned_topic"] == 27].shape[0] * 100
print(f"Percentage of situation code 140 in topic 27: {percent_in_topic_27:.2f}%")

Percentage of situation code 140 in topic 9: 71.96%
Percentage of situation code 140 in topic 18: 96.62%
Percentage of situation code 140 in topic 23: 95.34%
Percentage of situation code 140 in topic 27: 85.30%


In [ ]:
# how many percent of siuation code 140 are in topic 9, 18, 23, 27  


In [33]:
df_acc.head()

,accident_date,report_category,encoded_accident_situation,accident_situation,police_narrative,year,accident_id,main_situation_class,n_words,topic_main_0_3,...,VEJKATEGORI,VEJR,LYS,UHELDKOMMUNE,AAR,sprit_flag,has_cyclist,KODE_UHELDKOMMUNE,kommune_group,assigned_topic
0,1985-01-01,Anmsuh,242,Mødeuheld i øvrigt,part 1 overskrider spæreflade og påkører modkø...,1985,80001,2,9,0,...,Komv,Sne,Mørke,København,1985.0,1,0,101,Hovedstad,0
1,1985-01-01,Anmsuh,710,P. køretøj i h. vejside,1 der ukendt påkører den parkerede 2. 1 formen...,1985,80002,7,11,3,...,Komv,Sne,Mørke,København,1985.0,0,0,101,Hovedstad,3
2,1985-01-01,Pskduh,11,Ligeudkørsel til h.vejside,part 1 mister herredømmet over bilen og påkøre...,1985,80003,0,12,-1,...,Komv,Sne,Dagsl,København,1985.0,0,0,101,Hovedstad,-1
3,1985-01-01,Pskduh,811,Fodg. fra h. fortov,part 2 træder pludselig ud på kørebanen og påk...,1985,80004,8,12,0,...,Komv,Sne,Mørke,København,1985.0,1,0,101,Hovedstad,0
4,1985-01-01,Pskduh,241,Mødeuh. i el. 2.s kørebane,part 1 mister grundet glat føre herredømmet ov...,1985,80005,2,19,0,...,Komv,Sne,Mørke,København,1985.0,1,0,101,Hovedstad,0


In [33]:
# out of the accident siuation equal to 140, how many of those are in topic 0,1,2,3,4
accident_situation_counts = df_acc[df_acc["UHELDSSITUATION"] == 140]["assigned_topic"].value_counts(normalize=True) * 100
print(accident_situation_counts)

assigned_topic
-1     38.003792
 1     19.167664
 2     11.066742
 4      5.512186
 9      5.220901
 0      4.651040
 8      3.759176
 18     2.601447
 3      2.566493
 23     1.560233
 7      1.425712
 11     1.339915
 27     1.229756
 14     0.547617
 30     0.495715
 34     0.318826
 38     0.146173
 39     0.146173
 28     0.129225
 24     0.046606
 5      0.021184
 6      0.013770
 20     0.006355
 12     0.005296
 19     0.004237
 16     0.003178
 40     0.002118
 26     0.002118
 31     0.001059
 29     0.001059
 17     0.001059
 22     0.001059
 33     0.001059
 21     0.001059
Name: proportion, dtype: float64


In [31]:
from scipy.stats import chi2_contingency

rear_end_topics = {9: t9, 18: t18, 23: t23, 27: t27}
n_tests = len(rear_end_topics)  # 4 — én test per topic per felt

for topic, df in rear_end_topics.items():
    print(f"\n{'='*50}")
    print(f"TOPIC {topic} (n={len(df)})")
    print(f"{'='*50}")

    # --- VEJKATEGORI ---
    topic_counts = df['VEJKATEGORI'].value_counts()
    corpus_counts = df_acc['VEJKATEGORI'].value_counts()
    cats = corpus_counts.index
    observed = topic_counts.reindex(cats, fill_value=0).values
    corpus_prop = corpus_counts.reindex(cats, fill_value=0).values / corpus_counts.sum()
    expected = corpus_prop * topic_counts.sum()
    chi2, p_raw, dof, _ = chi2_contingency([observed, (corpus_prop * df_acc['VEJKATEGORI'].notna().sum()).astype(int)])
    p_corr = min(p_raw * n_tests, 1.0)

    lift = (df['VEJKATEGORI'].value_counts(normalize=True) /
            df_acc['VEJKATEGORI'].value_counts(normalize=True)).dropna().sort_values(ascending=False)

    print(f"\n  VEJKATEGORI — chi2={chi2:.1f}, p_bonferroni={p_corr:.4f}, dof={dof}")
    print(lift.round(3).to_string())

    # --- UHELDKOMMUNE lift ---
    topic_share = df['UHELDKOMMUNE'].value_counts(normalize=True)
    corpus_share = df_acc['UHELDKOMMUNE'].value_counts(normalize=True)
    lift_kom = (topic_share / corpus_share).dropna().sort_values(ascending=False)

    print(f"\n  UHELDKOMMUNE lift (top 20):")
    print(lift_kom.head(20).round(3).to_string())
    print()


TOPIC 9 (n=6850)

  VEJKATEGORI — chi2=1157.9, p_bonferroni=0.0000, dof=3
VEJKATEGORI
And. stat    3.181
Hldv         1.794
Komv         0.800
Fællesvej    0.470

  UHELDKOMMUNE lift (top 20):
UHELDKOMMUNE
Gladsaxe      2.154
Greve         2.130
Hørsholm      1.977
Solrød        1.647
Ishøj         1.623
Rudersdal     1.567
Aarhus        1.543
Halsnæs       1.535
Vallensbæk    1.486
Frederikss    1.423
Skanderbor    1.419
Kolding       1.399
Herlev        1.377
Favrskov      1.335
Assens        1.334
Hedensted     1.316
Odder         1.261
Roskilde      1.247
Middelfart    1.233
Brøndby       1.221


TOPIC 18 (n=2542)

  VEJKATEGORI — chi2=2866.2, p_bonferroni=0.0000, dof=3
VEJKATEGORI
And. stat    3.540
Hldv         3.107
Komv         0.471

  UHELDKOMMUNE lift (top 20):
UHELDKOMMUNE
Skanderbor    3.552
Middelfart    3.365
Brøndby       3.290
Solrød        2.731
Favrskov      2.698
Assens        2.661
Gladsaxe      2.383
Greve         2.212
Hedensted     2.197
Rødovre       2.114
Hor

In [34]:
# For a given topic, check road category distribution
t23 = df_acc[df_acc['assigned_topic'] == 23]
print(t23['VEJKATEGORI'].value_counts(normalize=True))
print(t23['UHELDKOMMUNE'].value_counts(normalize=True).head(20))


VEJKATEGORI
Hldv         0.761431
Komv         0.233930
And. stat    0.003976
Fællesvej    0.000663
Name: proportion, dtype: float64
UHELDKOMMUNE
Gladsaxe      0.076209
Aarhus        0.046388
Odense        0.046388
Brøndby       0.039761
Vejle         0.039761
Aalborg       0.039099
Køge          0.032472
Roskilde      0.027833
Gentofte      0.027170
København     0.025845
Glostrup      0.025182
Høje-Taast    0.025182
Lyngby-Taa    0.024520
Horsens       0.023857
Rudersdal     0.023857
Middelfart    0.023194
Kolding       0.022531
Skanderbor    0.021869
Randers       0.017893
Greve         0.017230
Name: proportion, dtype: float64


In [35]:
# For a given topic, check road category distribution
t9 = df_acc[df_acc['assigned_topic'] == 9]
print(t9['VEJKATEGORI'].value_counts(normalize=True))
print(t9['UHELDKOMMUNE'].value_counts(normalize=True).head(20))


VEJKATEGORI
Komv         0.629429
Hldv         0.358108
And. stat    0.006907
Fællesvej    0.005556
Name: proportion, dtype: float64
UHELDKOMMUNE
København     0.150751
Aarhus        0.060511
Odense        0.057508
Gladsaxe      0.028078
Aalborg       0.024625
Kolding       0.024174
Vejle         0.022222
Esbjerg       0.019670
Aabenraa      0.019069
Horsens       0.017568
Roskilde      0.017417
Slagelse      0.017267
Sønderborg    0.017267
Randers       0.017117
Greve         0.016066
Næstved       0.015766
Frederiksb    0.014414
Guldborgsu    0.012462
Skanderbor    0.012312
Køge          0.012012
Name: proportion, dtype: float64


In [37]:
t9 = df_acc[df_acc['assigned_topic'] == 9]

# Cross-tab of vejkategori within the suspected corridor municipalities
corridor = ['Tårnby', 'Hvidovre', 'Ishøj', 'Greve', 'Køge']
t9_corridor = t9[t9['UHELDKOMMUNE'].isin(corridor)]
print(t9_corridor['VEJKATEGORI'].value_counts(normalize=True))

# Compare against full topic distribution
print(t9['VEJKATEGORI'].value_counts(normalize=True))

VEJKATEGORI
Hldv         0.516224
Komv         0.430678
And. stat    0.029499
Fællesvej    0.023599
Name: proportion, dtype: float64
VEJKATEGORI
Komv         0.629429
Hldv         0.358108
And. stat    0.006907
Fællesvej    0.005556
Name: proportion, dtype: float64


In [40]:
# For a given topic, check road category distribution
t18 = df_acc[df_acc['assigned_topic'] == 18]
print(t18['VEJKATEGORI'].value_counts(normalize=True))
print(t18['UHELDKOMMUNE'].value_counts(normalize=True).head(20))


VEJKATEGORI
Hldv         0.621901
Komv         0.370248
And. stat    0.007851
Name: proportion, dtype: float64
UHELDKOMMUNE
København     0.078099
Aarhus        0.077273
Odense        0.059917
Aalborg       0.038017
Kolding       0.035537
Vejle         0.035537
Horsens       0.033471
Middelfart    0.031405
Gladsaxe      0.030579
Skanderbor    0.030165
Brøndby       0.028512
Køge          0.022727
Assens        0.021074
Roskilde      0.020661
Greve         0.017355
Rødovre       0.016942
Hedensted     0.016529
Favrskov      0.015289
Gentofte      0.015289
Næstved       0.014876
Name: proportion, dtype: float64


In [39]:
t18 = df_acc[df_acc['assigned_topic'] == 18]

# Cross-tab of vejkategori within the suspected corridor municipalities
corridor = ['Tårnby', 'Hvidovre', 'Ishøj', 'Greve', 'Køge']
t18_corridor = t18[t18['UHELDKOMMUNE'].isin(corridor)]
print(t18_corridor['VEJKATEGORI'].value_counts(normalize=True))

# Compare against full topic distribution
print(t9['VEJKATEGORI'].value_counts(normalize=True))

VEJKATEGORI
Hldv         0.740260
Komv         0.220779
And. stat    0.038961
Name: proportion, dtype: float64
VEJKATEGORI
Komv         0.629429
Hldv         0.358108
And. stat    0.006907
Fællesvej    0.005556
Name: proportion, dtype: float64


In [42]:
t18 = df_acc[df_acc['assigned_topic'] == 18]
print("VEJKATEGORI")
print(t18['VEJKATEGORI'].value_counts(normalize=True))
print()
print("UHELDKOMMUNE")
print(t18['UHELDKOMMUNE'].value_counts(normalize=True).head(20))


VEJKATEGORI
VEJKATEGORI
Hldv         0.621901
Komv         0.370248
And. stat    0.007851
Name: proportion, dtype: float64

UHELDKOMMUNE
UHELDKOMMUNE
København     0.078099
Aarhus        0.077273
Odense        0.059917
Aalborg       0.038017
Kolding       0.035537
Vejle         0.035537
Horsens       0.033471
Middelfart    0.031405
Gladsaxe      0.030579
Skanderbor    0.030165
Brøndby       0.028512
Køge          0.022727
Assens        0.021074
Roskilde      0.020661
Greve         0.017355
Rødovre       0.016942
Hedensted     0.016529
Favrskov      0.015289
Gentofte      0.015289
Næstved       0.014876
Name: proportion, dtype: float64


In [27]:
# For a given topic, check road category distribution
t27 = df_acc[df_acc['assigned_topic'] == 27]
print(t27['VEJKATEGORI'].value_counts(normalize=True))
print(t27['UHELDKOMMUNE'].value_counts(normalize=True).head(20))
print(t27['KOMMUNE_KODE'].value_counts(normalize=True).head(20))



VEJKATEGORI
Hldv         0.531227
Komv         0.464364
And. stat    0.004409
Name: proportion, dtype: float64
UHELDKOMMUNE
Odense        0.054372
Gladsaxe      0.045555
Aarhus        0.041881
Aalborg       0.035268
København     0.034533
Randers       0.034533
Gentofte      0.033799
Glostrup      0.029390
Brøndby       0.027921
Middelfart    0.024247
Rudersdal     0.022043
Vejle         0.022043
Roskilde      0.019104
Høje-Taast    0.019104
Horsens       0.018369
Næstved       0.018369
Kolding       0.018369
Aabenraa      0.018369
Slagelse      0.016899
Hillerød      0.016165
Name: proportion, dtype: float64


KeyError: 'KOMMUNE_KODE'

In [35]:
import plotly.express as px

In [55]:
topics_to_investigate = [9, 18, 23, 27]

data_filtered = df_acc[df_acc["assigned_topic"].isin(topics_to_investigate)].copy()

data_filtered["year"] = pd.Categorical(
    data_filtered["year"].astype(str),
    categories=[str(y) for y in sorted(data_filtered["year"].unique())],
    ordered=True
)

fig = px.scatter_mapbox(
    data_filtered,
    lat="y",
    lon="x",
    animation_frame="year",
    color="assigned_topic",
    hover_data=[
        "assigned_topic",
        "accident_id"
    ],
    zoom=6,
    height=900,
    title="Accidents by BERTopic Topic"
)

fig.update_layout(
    mapbox_style="carto-positron"
)

fig.show(renderer="browser")

/tmp/ipykernel_1166409/537455980.py:11: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [ ]:
topics_to_investigate = [9, 18, 23, 27]

data_filtered = df_acc[df_acc["assigned_topic"].isin(topics_to_investigate)].copy()

data_filtered["year"] = pd.Categorical(
    data_filtered["year"].astype(str),
    categories=[str(y) for y in sorted(data_filtered["year"].unique())],
    ordered=True
)

fig = px.scatter_mapbox(
    data_filtered,
    lat="y",
    lon="x",
    animation_frame="year",
    color="assigned_topic",
    hover_data=[
        "assigned_topic",
        "accident_id"
    ],
    zoom=6,
    height=900,
    title="Accidents by BERTopic Topic"
)

fig.update_layout(
    mapbox_style="carto-positron"
)

fig.show(renderer="browser")

In [37]:
topics_to_investigate = [18]

color_map = {
    "18": "#C2185B",   # dark pink
    "9": "#1565C0",  # mid blue
    "23": "#2E7D32",  # dark green
    "27": "#E65100",  # dark orange
}

data_filtered = df_acc[df_acc["assigned_topic"].isin(topics_to_investigate)].copy()

data_filtered["assigned_topic"] = data_filtered["assigned_topic"].astype(str)

data_filtered["year"] = pd.Categorical(
    data_filtered["year"].astype(str),
    categories=[str(y) for y in sorted(data_filtered["year"].unique())],
    ordered=True
)

fig = px.scatter_mapbox(
    data_filtered,
    lat="y",
    lon="x",
    animation_frame="year",
    color="assigned_topic",
    color_discrete_map=color_map,
    hover_data=[
        "assigned_topic",
        "accident_id"
    ],
    zoom=6,
    height=900,
    title="Accidents by BERTopic Topic — Accidents in Topics 18 Highlighted"
)

fig.update_layout(
    mapbox_style="carto-positron"
)

fig.show(renderer="browser")


/tmp/ipykernel_3026201/3389357474.py:20: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [75]:
topics_to_investigate = [18]

color_map = {
    "9": "#C2185B",   # dark pink
    "18": "#1565C0",  # mid blue
    "23": "#2E7D32",  # dark green
    "27": "#E65100",  # dark orange
}

data_filtered = df_acc[df_acc["assigned_topic"].isin(topics_to_investigate)].copy()

data_filtered["assigned_topic"] = data_filtered["assigned_topic"].astype(str)

data_filtered["year"] = pd.Categorical(
    data_filtered["year"].astype(str),
    categories=[str(y) for y in sorted(data_filtered["year"].unique())],
    ordered=True
)

fig = px.scatter_mapbox(
    data_filtered,
    lat="y",
    lon="x",
    animation_frame="year",
    color="assigned_topic",
    color_discrete_map=color_map,
    hover_data=[
        "assigned_topic",
        "accident_id"
    ],
    zoom=6,
    height=900,
    title="Accidents by BERTopic Topic — Accidents in Topics 18 Highlighted"
)

fig.update_layout(
    mapbox_style="carto-positron"
)

fig.show(renderer="browser")


/tmp/ipykernel_1166409/1816396232.py:20: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [74]:
topics_to_investigate = [27]

color_map = {
    "9": "#C2185B",   # dark pink
    "18": "#1565C0",  # mid blue
    "23": "#2E7D32",  # dark green
    "27": "#E65100",  # dark orange
}

data_filtered = df_acc[df_acc["assigned_topic"].isin(topics_to_investigate)].copy()

data_filtered["assigned_topic"] = data_filtered["assigned_topic"].astype(str)

data_filtered["year"] = pd.Categorical(
    data_filtered["year"].astype(str),
    categories=[str(y) for y in sorted(data_filtered["year"].unique())],
    ordered=True
)

fig = px.scatter_mapbox(
    data_filtered,
    lat="y",
    lon="x",
    animation_frame="year",
    color="assigned_topic",
    color_discrete_map=color_map,
    hover_data=[
        "assigned_topic",
        "accident_id"
    ],
    zoom=6,
    height=900,
    title="Accidents by BERTopic Topic — Accidents in Topics 27 Highlighted"
)

fig.update_layout(
    mapbox_style="carto-positron"
)

fig.show(renderer="browser")


/tmp/ipykernel_1166409/2727679451.py:20: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [61]:
data_filtered = df_acc[df_acc["accident_situation"] == 140].copy()

data_filtered["year"] = pd.Categorical(
    data_filtered["year"].astype(str),
    categories=[str(y) for y in sorted(data_filtered["year"].unique())],
    ordered=True
)

fig = px.scatter_mapbox(
    data_filtered,
    lat="y",
    lon="x",
    animation_frame="year",
    color="assigned_topic",
    hover_data=[
        "assigned_topic",
        "accident_id"
    ],
    zoom=6,
    height=900,
    title="Accidents by BERTopic Topic — Situation Class 1400"
)

fig.update_layout(
    mapbox_style="carto-positron"
)

fig.show(renderer="browser")

/tmp/ipykernel_1166409/3382202584.py:9: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


KeyboardInterrupt: 

In [10]:
topics_to_investigate = [9, 18, 23, 27]

for t in topics_to_investigate:
    subset = df[df['topic_main_0_3'] == t]['probability_main_0_3']
    print(f"\nTopic {t} — n={len(subset)}")
    print(f"  Mean:    {subset.mean():.3f}")
    print(f"  Median:  {subset.median():.3f}")
    print(f"  Std:     {subset.std():.3f}")
    print(f"  >0.7:    {(subset > 0.7).mean():.1%}")
    print(f"  >0.5:    {(subset > 0.5).mean():.1%}")
    print(f"  <0.3:    {(subset < 0.3).mean():.1%}")


Topic 9 — n=6850
  Mean:    0.838
  Median:  0.834
  Std:     0.126
  >0.7:    78.6%
  >0.5:    100.0%
  <0.3:    0.0%

Topic 18 — n=2542
  Mean:    0.804
  Median:  1.000
  Std:     0.259
  >0.7:    69.2%
  >0.5:    79.3%
  <0.3:    7.4%

Topic 23 — n=1545
  Mean:    0.827
  Median:  1.000
  Std:     0.247
  >0.7:    71.8%
  >0.5:    83.4%
  <0.3:    4.2%

Topic 27 — n=1361
  Mean:    0.943
  Median:  1.000
  Std:     0.098
  >0.7:    96.5%
  >0.5:    100.0%
  <0.3:    0.0%


In [11]:
for t in topics_to_investigate:
    row = topic_main[topic_main['Topic'] == t].iloc[0]
    print(f"\nTopic {t} — {row['Name']}")
    print(f"  Count: {row['Count']}")
    print(f"  Keywords: {row['Representation']}")


Topic 9 — 9_bagfra_kø_påkørt bagfra_skubbet
  Count: 6850
  Keywords: ['bagfra', 'kø', 'påkørt bagfra', 'skubbet', 'bremsede', 'køkørsel', 'stille', 'forankørende', 'holdt stille', 'påkøres bagfra', 'bagfra skubbes', 'bagfra bremsede', 'bagfra skubbet', 'færdsel påkøres', 'ringe', 'skubbes', 'bremser', 'påkøres', 'skubbet frem', 'bagfra bagfra']

Topic 18 — 18_harmonikasammenstød_harmonika_harmonika sammenstød_harmonikasammenstød biler
  Count: 2542
  Keywords: ['harmonikasammenstød', 'harmonika', 'harmonika sammenstød', 'harmonikasammenstød biler', 'biler harmonikasammenstød', 'biler', 'personbiler', 'harmonikasammenstød harmonikasammenstød', 'hamonikasammenstød', 'skubbet', 'harmonikasammenstød vognbane', 'harmonikasammenstød personbiler', 'skete harmonikasammenstød', 'kø', 'skubbet harmonikasammenstød', 'impliceret', 'sammenstød', 'køkørsel', 'motorvej', 'trafik']

Topic 23 — 23_kødannelse_kødannelse bremsede_bremsede kødannelse_kødannelse bagfra
  Count: 1545
  Keywords: ['kødanne

In [14]:
import ast

sets = {}
for t in [9, 18, 23, 27]:
    rep = topic_main[topic_main['Topic'] == t]['Representation'].iloc[0]
    # Parse string representation of list into actual list
    if isinstance(rep, str):
        rep = ast.literal_eval(rep)
    sets[t] = set(rep)

for t1 in [9, 18, 23, 27]:
    for t2 in [9, 18, 23, 27]:
        if t1 < t2:
            intersection = sets[t1] & sets[t2]
            union = sets[t1] | sets[t2]
            jaccard = len(intersection) / len(union)
            print(f"Topic {t1} ∩ Topic {t2}: {intersection}")
            print(f"  Jaccard: {jaccard:.3f}")

Topic 9 ∩ Topic 18: {'kø', 'køkørsel', 'skubbet'}
  Jaccard: 0.081
Topic 9 ∩ Topic 23: {'bagfra', 'bremsede', 'skubbet'}
  Jaccard: 0.081
Topic 9 ∩ Topic 27: {'bagfra', 'forankørende', 'påkørt bagfra'}
  Jaccard: 0.081
Topic 18 ∩ Topic 23: {'skubbet'}
  Jaccard: 0.026
Topic 18 ∩ Topic 27: set()
  Jaccard: 0.000
Topic 23 ∩ Topic 27: {'bagfra'}
  Jaccard: 0.026


In [30]:
# compare max_element_number for topic 9 vs topic 18
t9 = df_acc[df_acc['assigned_topic'] == 9]
t18 = df_acc[df_acc['assigned_topic'] == 18]
t23 = df_acc[df_acc['assigned_topic'] == 23]
t27 = df_acc[df_acc['assigned_topic'] == 27]

#print distribution (average, mean, 25%, 50%, 75%, 95%) of max_element_number for topic 9
print("Topic 9 - max_element_number distribution:")
print("Average:", t9['max_element_number'].mean())
print("Mean:", t9['max_element_number'].median())
print("25%:", t9['max_element_number'].quantile(0.25))
print("50%:", t9['max_element_number'].quantile(0.5))
print("75%:", t9['max_element_number'].quantile(0.75))
print("95%:", t9['max_element_number'].quantile(0.95))
print()
#print distribution (average, mean, 25%, 50%, 75%, 95%) of max_element_number for topic 18
print("Topic 18 - max_element_number distribution:")
print("Average:", t18['max_element_number'].mean())
print("Mean:", t18['max_element_number'].median())
print("25%:", t18['max_element_number'].quantile(0.25))
print("50%:", t18['max_element_number'].quantile(0.5))
print("75%:", t18['max_element_number'].quantile(0.75))
print("95%:", t18['max_element_number'].quantile(0.95))
#print distribution (average, mean, 25%, 50%, 75%, 95%) of max_element_number for topic 23
print("Topic 23 - max_element_number distribution:")
print("Average:", t23['max_element_number'].mean())
print("Mean:", t23['max_element_number'].median())
print("25%:", t23['max_element_number'].quantile(0.25))
print("50%:", t23['max_element_number'].quantile(0.5))
print("75%:", t23['max_element_number'].quantile(0.75))
print("95%:", t23['max_element_number'].quantile(0.95))
#print distribution (average, mean, 25%, 50%, 75%, 95%) of max_element_number for topic 27
print("Topic 27 - max_element_number distribution:")
print("Average:", t27['max_element_number'].mean())
print("Mean:", t27['max_element_number'].median())
print("25%:", t27['max_element_number'].quantile(0.25))
print("50%:", t27['max_element_number'].quantile(0.5))
print("75%:", t27['max_element_number'].quantile(0.75))
print("95%:", t27['max_element_number'].quantile(0.95))


Topic 9 - max_element_number distribution:
Average: 2.285029119561494
Mean: 2.0
25%: 2.0
50%: 2.0
75%: 2.0
95%: 4.0

Topic 18 - max_element_number distribution:
Average: 3.733510402833112
Mean: 3.0
25%: 3.0
50%: 3.0
75%: 4.0
95%: 6.0
Topic 23 - max_element_number distribution:
Average: 2.5925349922239502
Mean: 2.0
25%: 2.0
50%: 2.0
75%: 3.0
95%: 4.0
Topic 27 - max_element_number distribution:
Average: 2.389041095890411
Mean: 2.0
25%: 2.0
50%: 2.0
75%: 3.0
95%: 4.0


In [28]:
from scipy.stats import chi2_contingency

rear_end_topics = [9, 18, 23, 27]
age_group_cols = ["element_1_age_group", "element_2_age_group"]
n_tests = len(rear_end_topics) * len(age_group_cols)  # 8

for topic in rear_end_topics:
    t = df_acc[df_acc["assigned_topic"] == topic]
    print(f"\n{'='*50}")
    print(f"TOPIC {topic} (n={len(t)})")
    print(f"{'='*50}")
    
    for col in age_group_cols:
        topic_counts = t[col].value_counts()
        corpus_counts = df_acc[col].value_counts()
        
        # Align on same categories
        cats = corpus_counts.index
        observed = topic_counts.reindex(cats, fill_value=0).values
        expected_prop = corpus_counts.reindex(cats, fill_value=0).values / corpus_counts.sum()
        expected = expected_prop * topic_counts.sum()
        
        chi2, p_raw, dof, _ = chi2_contingency(
            [observed, (expected_prop * df_acc[col].notna().sum()).astype(int)]
        )
        p_corr = min(p_raw * n_tests, 1.0)
        
        topic_share = topic_counts / topic_counts.sum()
        baseline_share = corpus_counts / corpus_counts.sum()
        lift = (topic_share / baseline_share).dropna().sort_values(ascending=False)
        
        print(f"\n  {col}:")
        print(f"  chi2={chi2:.1f}, p_raw={p_raw:.4f}, p_bonferroni={p_corr:.4f}, dof={dof}")
        print(lift.round(3).to_string())
    print()



TOPIC 9 (n=6850)

  element_1_age_group:
  chi2=132.4, p_raw=0.0000, p_bonferroni=0.0000, dof=6
element_1_age_group
26-35      1.161
36-45      1.123
21-25      1.082
46-60      0.989
0-20       0.862
Unknown    0.819
61+        0.696

  element_2_age_group:
  chi2=383.6, p_raw=0.0000, p_bonferroni=0.0000, dof=6
element_2_age_group
36-45      1.253
46-60      1.216
26-35      1.121
21-25      0.865
61+        0.784
0-20       0.513
Unknown    0.000


TOPIC 18 (n=2542)

  element_1_age_group:
  chi2=122.3, p_raw=0.0000, p_bonferroni=0.0000, dof=6
element_1_age_group
26-35      1.223
36-45      1.201
46-60      1.109
21-25      1.009
0-20       0.671
61+        0.636
Unknown    0.000

  element_2_age_group:
  chi2=186.3, p_raw=0.0000, p_bonferroni=0.0000, dof=6
element_2_age_group
36-45      1.245
46-60      1.236
26-35      1.170
61+        0.860
21-25      0.752
0-20       0.452
Unknown    0.000


TOPIC 23 (n=1545)

  element_1_age_group:
  chi2=68.2, p_raw=0.0000, p_bonferroni=0.0000

In [36]:
import re
t9 = df_acc[df_acc['assigned_topic'] == 9]
t18 = df_acc[df_acc['assigned_topic'] == 18]
t23 = df_acc[df_acc['assigned_topic'] == 23]
t27 = df_acc[df_acc['assigned_topic'] == 27]

search_terms = {
    'Vehicle type': ['elbil', 'hybrid', 'lastbil', 'varebil', 'motorcykel', 'cyklist'],
    'Distraction': ['mobiltelefon', 'mobil','tablet', 'telefon', 'distraheret', 'uopmærksom', 'så ikke', 'overså'],
    'Fatigue': ['træt', 'søvn', 'faldt i søvn'],
    'Weather': ['regn', 'sne', 'tåge', 'glat']
}

rear_end_topics = {'T9': t9, 'T18': t18, 'T23': t23, 'T27': t27}
text_col = 'police_narrative'  # replace with your actual column name

for category, terms in search_terms.items():
    print(f"\n{'='*50}")
    print(f"{category}")
    print(f"{'='*50}")
    for term in terms:
        print(f"\n  '{term}':")
        for name, df in rear_end_topics.items():
            count = df[text_col].str.contains(term, case=False, na=False).sum()
            pct = count / len(df) * 100
            print(f"    {name}: {count} ({pct:.2f}%)")


Vehicle type

  'elbil':
    T9: 0 (0.00%)
    T18: 2 (0.08%)
    T23: 0 (0.00%)
    T27: 0 (0.00%)

  'hybrid':
    T9: 0 (0.00%)
    T18: 0 (0.00%)
    T23: 0 (0.00%)
    T27: 0 (0.00%)

  'lastbil':
    T9: 290 (4.23%)
    T18: 76 (2.99%)
    T23: 36 (2.33%)
    T27: 43 (3.16%)

  'varebil':
    T9: 143 (2.09%)
    T18: 84 (3.30%)
    T23: 62 (4.01%)
    T27: 40 (2.94%)

  'motorcykel':
    T9: 35 (0.51%)
    T18: 4 (0.16%)
    T23: 5 (0.32%)
    T27: 14 (1.03%)

  'cyklist':
    T9: 45 (0.66%)
    T18: 7 (0.28%)
    T23: 1 (0.06%)
    T27: 21 (1.54%)

Distraction

  'mobiltelefon':
    T9: 4 (0.06%)
    T18: 1 (0.04%)
    T23: 1 (0.06%)
    T27: 0 (0.00%)

  'mobil':
    T9: 4 (0.06%)
    T18: 1 (0.04%)
    T23: 1 (0.06%)
    T27: 1 (0.07%)

  'tablet':
    T9: 0 (0.00%)
    T18: 0 (0.00%)
    T23: 0 (0.00%)
    T27: 0 (0.00%)

  'telefon':
    T9: 21 (0.31%)
    T18: 6 (0.24%)
    T23: 2 (0.13%)
    T27: 1 (0.07%)

  'distraheret':
    T9: 1 (0.01%)
    T18: 1 (0.04%)
    T23: 1 

In [39]:
terms_to_check = ['uopmærksom', 'overså', 'glat', 'regn', 'sne', 'lastbil', 'varebil','telefon']

text_col = 'police_narrative'  # replace with your actual column name

print(f"{'Term':<20} {'Corpus':>10} {'T9':>10} {'T18':>10} {'T23':>10} {'T27':>10}")
print("=" * 70)

for term in terms_to_check:
    corpus_pct = df_acc[text_col].str.contains(term, case=False, na=False).mean() * 100
    t9_pct  = t9[text_col].str.contains(term, case=False, na=False).mean() * 100
    t18_pct = t18[text_col].str.contains(term, case=False, na=False).mean() * 100
    t23_pct = t23[text_col].str.contains(term, case=False, na=False).mean() * 100
    t27_pct = t27[text_col].str.contains(term, case=False, na=False).mean() * 100
    print(f"{term:<20} {corpus_pct:>9.2f}% {t9_pct:>9.2f}% {t18_pct:>9.2f}% {t23_pct:>9.2f}% {t27_pct:>9.2f}%")


Term                     Corpus         T9        T18        T23        T27
uopmærksom                1.37%      2.79%      1.77%      5.11%      1.84%
overså                    6.45%      5.58%      4.92%     11.46%      4.04%
glat                      2.47%      1.66%      0.98%      0.58%      2.13%
regn                      0.94%      0.44%      0.63%      0.13%      1.10%
sne                       0.92%      0.42%      0.51%      0.26%      1.03%
lastbil                   3.64%      4.23%      2.99%      2.33%      3.16%
varebil                   3.25%      2.09%      3.30%      4.01%      2.94%
telefon                   0.58%      0.31%      0.24%      0.13%      0.07%


In [34]:
from collections import Counter
import re

# Load stopwords
with open('/home/s215005/traffic_project/Thesis-traffic-safety/final_stopwords.txt', 'r', encoding='utf-8') as f:
    stopwords = set(f.read().splitlines())

# Manual additions — artifacts and filler words only
# Manual additions — artifacts and filler words only
manual_exclude = {
    # Artifacts
    's', 'p', 'a', 'f', 'g', 'm',
    # Generic verb forms
    'skete', 'foretog', 'kører', 'køre', 'ramte', 'ramt',
    'nåede', 'standsede', 'standset', 'bemærkede', 'opdagede',
    'undgå', 'foretage',
    # Generic filler nouns
    'parterne', 'herunder', 'side', 'frem', 'hvilket',
    'uheldet', 'færdselsuheld', 'hinanden', 'ligeledes',
    'umiddelbart', 'hvorunder', 'involveret', 'opstod', 'impliceret',
    # Generic spatial/directional
    'øst', 'venstre', 'højre', 'nordgående', 'sydgående',
}


stopwords = stopwords | manual_exclude


In [35]:
text_col = 'police_narrative'  # replace with your actual column name

rear_end_topics = {'T9': t9, 'T18': t18, 'T23': t23, 'T27': t27}

for name, df in rear_end_topics.items():
    print(f"\n{'='*50}")
    print(f"{name} (n={len(df)}) — top 40 words")
    print(f"{'='*50}")
    
    # Combine all narratives, lowercase, tokenise
    all_text = ' '.join(df[text_col].dropna().str.lower().values)
    words = re.findall(r'\b[a-zæøå]+\b', all_text)
    
    # Remove stopwords
    filtered = [w for w in words if w not in stopwords]
    
    # Count and print
    counts = Counter(filtered).most_common(40)
    for word, count in counts:
        pct = count / len(df) * 100
        print(f"  {word:<30} {count:>6}  ({pct:.1f}%)")



T9 (n=6850) — top 40 words
  bagfra                           2918  (42.6%)
  holdt                            1451  (21.2%)
  påkørt                           1236  (18.0%)
  stille                           1050  (15.3%)
  bremsede                         1021  (14.9%)
  skubbet                           787  (11.5%)
  ringe                             653  (9.5%)
  forankørende                      525  (7.7%)
  kø                                446  (6.5%)
  overhaling                        420  (6.1%)
  overså                            384  (5.6%)
  køretøj                           381  (5.6%)
  påkører                           345  (5.0%)
  bremse                            332  (4.8%)
  lastbil                           301  (4.4%)
  køkørsel                          246  (3.6%)
  trafik                            245  (3.6%)
  biler                             221  (3.2%)
  påkøres                           219  (3.2%)
  motorvejen                        219  (3.2%)
  svin